# Lab 10 — Odometry vs. odometry + gyro localization

Run **A**: `course = course_odom` · Run **B**: `course = course + gyro_inc`.
Same program (`Lab10_odometry_gyro.qrs`), same controller (turn-then-drive), only `MODE` changes.

**Log format** (one line per ~10 ms control cycle, written by the robot to `lab10_log.txt`):

`t_ms, ts_us, dt_ms, phase, i, encL_deg, encR_deg, gz_mdps, x_cm, y_cm, course, course_odom, course_gyro, u_l, u_r`

* `phase`: 1 = turn in place, 2 = drive
* `i`: index of the current target waypoint (segment `k = i` goes from waypoint `i-1` to `i`)
* `encL_deg = encoder E4`, `encR_deg = encoder E3` are the raw encoder angles in degrees, so `theta = enc * pi/180` (logged raw so no precision is lost)
* every run starts with a `# RUN MODE=…` header and a `# b0_mdps=…` line (b0 = mean gyro rate of the 10 s rest).

Run A (`charos_Lab10_runA_odometry.qrs`) writes `charos_lab10_runA.txt`, run B (`charos_Lab10_runB_gyro.qrs`) writes `charos_lab10_runB.txt`. Download both from the robot.

**How to use:** run the cells top to bottom. Upload both logs when asked. Fill in the floor measurements in Section 6.

In [ ]:
import math, io, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# ======== your parameters ========
DIGITS = "70"                 # last two digits of your student ID (same as in the .qrs)
W, H, GAP = [40.0, 60.0], 80.0, 20.0  # widths of digit 1 and 2, height, gap, cm (same as in the .qrs)
r, B = 4.3, 19.8              # TRUE wheel radius and wheel base of this robot, cm (measured)
# The robot's own estimate is checked with the r, B written in each log's '# RUN' header;
# all offline results use the r, B above.
GYRO_SIGN = 1                 # Lab 9 §6: positive gyro rate = CCW
S_CCW, S_CW = 0.0, 0.0        # gyro scale factor used offline (same as on the robot). Lab 9 gave +11.99 % / +8.98 %,
                              # but it was measured against encoders computed with the wrong r/B (2.8/15.4).
W_MAX = 286.4                 # Lab 9 §3 gyro range, deg/s
# The robot itself ran with s = 0 (see the '# RUN' header). The robot's own estimate is checked
# with the header values; every offline gyro course below uses the Lab 9 values above.
LOG_FILES = ["charos_lab10_runA.txt", "charos_lab10_runB.txt"]   # uploaded in Colab if not found

## 1. Ideal trajectory (Step 1)

In [ ]:
STROKES = {
    "0": ["BL", "TL", "TR", "BR", "BL"],
    "1": ["BR", "TR"],
    "2": ["TL", "TR", "MR", "ML", "BL", "BR"],
    "3": ["TL", "TR", "MR", "ML", "MR", "BR", "BL"],
    "4": ["TL", "ML", "MR", "TR", "BR"],
    "5": ["TR", "TL", "ML", "MR", "BR", "BL"],
    "6": ["TR", "TL", "BL", "BR", "MR", "ML"],
    "7": ["TL", "TR", "BR"],
    "8": ["ML", "TL", "TR", "BR", "BL", "ML", "MR"],
    "9": ["MR", "ML", "TL", "TR", "BR", "BL"],
}
def _corner(n, W, H):
    return {"TL": (0, H), "TR": (W, H), "ML": (0, H/2), "MR": (W, H/2), "BL": (0, 0), "BR": (W, 0)}[n]

def digit_waypoints(digits, W=40.0, H=80.0, GAP=20.0):
    pts, owner = [], []
    widths = W if isinstance(W, (list, tuple)) else [W] * len(str(digits))
    ox = 0.0
    for k, d in enumerate(str(digits)):
        for c in STROKES[d]:
            px, py = _corner(c, widths[k], H); pts.append((ox + px, py)); owner.append(k)
        ox += widths[k] + GAP
    clean, own = [pts[0]], [owner[0]]
    for p, o in zip(pts[1:], owner[1:]):
        if math.dist(p, clean[-1]) < 1e-9: continue
        if len(clean) >= 2:
            a, b = clean[-2], clean[-1]
            h1 = math.atan2(b[1]-a[1], b[0]-a[0]); h2 = math.atan2(p[1]-b[1], p[0]-b[0])
            if abs(math.remainder(h2-h1, 2*math.pi)) < 1e-9 and own[-1] == o:
                clean[-1], own[-1] = p, o; continue
        clean.append(p); own.append(o)
    x0, y0 = clean[0]
    X = np.array([p[0]-x0 for p in clean]); Y = np.array([p[1]-y0 for p in clean])
    labels = ["transit" if own[i] != own[i+1] else f"digit {own[i+1]+1}" for i in range(len(X)-1)]
    return X, Y, labels

X_ref, Y_ref, seg_label = digit_waypoints(DIGITS, W, H, GAP)
NSEG = len(X_ref) - 1
course_ref = np.arctan2(np.diff(Y_ref), np.diff(X_ref))
path_ref = np.hypot(np.diff(X_ref), np.diff(Y_ref))

wrap = lambda a: (np.asarray(a) + np.pi) % (2*np.pi) - np.pi
# continuous (unwrapped) reference: the robot always turns the short way
course_ref_cont = np.zeros(NSEG); prev = 0.0
for k in range(NSEG):
    prev = prev + wrap(course_ref[k] - prev); course_ref_cont[k] = prev

ref = pd.DataFrame({"seg": np.arange(1, NSEG+1), "part": seg_label,
                    "from": [f"({X_ref[k]:.0f},{Y_ref[k]:.0f})" for k in range(NSEG)],
                    "to": [f"({X_ref[k+1]:.0f},{Y_ref[k+1]:.0f})" for k in range(NSEG)],
                    "course_ref_deg": np.degrees(course_ref).round(1), "path_ref_cm": path_ref.round(1)})
print("X_ref =", "{" + ", ".join(f"{v:g}" for v in X_ref) + "}")
print("Y_ref =", "{" + ", ".join(f"{v:g}" for v in Y_ref) + "}")
display(ref) if "display" in globals() else print(ref.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(X_ref, Y_ref, "k-o", ms=4)
for k in range(NSEG):
    ax.annotate(str(k+1), ((X_ref[k]+X_ref[k+1])/2, (Y_ref[k]+Y_ref[k+1])/2), color="C3", fontsize=9)
ax.plot(0, 0, "g^", ms=12, label="start pose (facing +x)")
ax.set_aspect("equal"); ax.set_xlabel("x, cm"); ax.set_ylabel("y, cm"); ax.set_title(f"Ideal trajectory, digits {DIGITS}"); ax.legend()
plt.show()

## 2. Load the logs

In [ ]:
COLS = ["t_ms","ts_us","dt_ms","phase","i","encL_deg","encR_deg","gz_mdps",
        "x_cm","y_cm","course","course_odom","course_gyro","u_l","u_r"]

import os
if not LOG_FILES or not all(os.path.exists(f) for f in LOG_FILES):
    try:
        from google.colab import files
        print("Upload charos_lab10_runA.txt and charos_lab10_runB.txt")
        LOG_FILES = list(files.upload().keys())
    except ImportError:
        raise SystemExit("Put the log files next to the notebook or set LOG_FILES.")

def parse_runs(paths):
    runs = []
    for p in paths:
        cur = None
        for line in open(p, encoding="utf-8", errors="replace"):
            line = line.strip()
            if not line: continue
            if line.startswith("# RUN"):
                cur = {"file": p, "header": dict(re.findall(r"(\w+)=(\S+)", line)), "rows": [], "b0": None}
                runs.append(cur); continue
            if cur is None: continue
            if line.startswith("# b0_mdps="):
                cur["b0"] = float(line.split("=")[1].split()[0]); continue
            if line.startswith("#"): continue
            v = line.split(",")
            if len(v) == len(COLS):
                try: cur["rows"].append([float(a) for a in v])
                except ValueError: pass
    out = []
    for k, R in enumerate(runs):
        df = pd.DataFrame(R["rows"], columns=COLS)
        if df.empty: continue
        mode = int(float(R["header"].get("MODE", -1)))
        out.append(dict(idx=k, mode=mode, df=df, b0=R["b0"], header=R["header"], file=R["file"],
                        complete=int(df.i.max()) >= NSEG, last_i=int(df.i.max())))
    return out

all_runs = parse_runs(LOG_FILES)
for R in all_runs:
    print(f"run #{R['idx']}: MODE={R['mode']}  rows={len(R['df'])}  b0={R['b0']}  reached i={R['last_i']}  complete={R['complete']}  ({R['file']})")

def pick(mode):
    c = [R for R in all_runs if R["mode"] == mode and R["complete"]]
    if not c: raise ValueError(f"no complete run with MODE={mode}")
    return c[-1]   # last complete run = the one you kept
runs = {"A": pick(0), "B": pick(1)}
for name, R in runs.items():
    df = R["df"]
    df["t"] = (df.t_ms - df.t_ms.iloc[0]) / 1000
    df["phase"] = df.phase.round().astype(int)          # older program version logged 1.99999 etc.
    # dt check: the gyro timestamp is NOT in microseconds on every TRIK (one robot gave ~3.9 units/ms).
    k = (df.ts_us.iloc[-1] - df.ts_us.iloc[0]) / (df.t_ms.iloc[-1] - df.t_ms.iloc[0])   # timestamp units per ms
    dt_ts = np.concatenate([[df.dt_ms.iloc[0]], np.diff(df.ts_us.values) / k])
    R["k_ts"] = k
    if abs(np.median(df.dt_ms) / np.median(dt_ts) - 1) > 0.2:
        print(f"run {name}: logged dt_ms (median {np.median(df.dt_ms):.3f} ms) does not match the gyro timestamp "
              f"({k:.3f} units/ms -> median {np.median(dt_ts):.1f} ms). dt is recomputed from the timestamp; "
              f"the on-robot course_gyro of this run is wrong by a factor {np.median(dt_ts) / np.median(df.dt_ms):.0f}.")
        R["dt_fixed"] = True
        df["dt_ms"] = dt_ts
    else:
        R["dt_fixed"] = False
    print(f"run {name}: using run #{R['idx']}, {R['df'].t.iloc[-1]:.1f} s, "
          f"mean cycle {R['df'].dt_ms[R['df'].phase > 0].mean():.1f} ms")

**Note on the parameters.** The robot uses r = 4.3 cm and B = 19.8 cm (measured on this robot) and the gyro with `s = 0`: Lab 9's scale factor (+11.99 % / +8.98 %) was measured against encoders computed with the wrong wheel size (r = 2.8 cm, B = 15.4 cm), and an earlier Lab 10 attempt with that wrong size showed that with the true r/B the gyro and encoders agree within a few %. `W_MAX` = 286.4 °/s (Lab 9). The bias `b0` is the mean of the 10 s rest before each run, as the handout allows. The values the robot actually used are in each log's `# RUN` header.

**Deviations from the handout.** The control loop runs at about 55 ms per cycle instead of about 10 ms (speed of the TRIK block interpreter). The gyro timestamp `gyroscope[3]` counts in 256 µs ticks (about 3.90 per ms, as in Lab 9); the program measures this rate during the rest. Encoder angles are logged in degrees (θ = deg·π/180).

## 3. Offline recomputation (Step 4.2)

Both localization methods are recomputed from the raw columns of **both** logs (`encL_deg, encR_deg, gz_mdps, dt_ms`). This gives 4 paths. Recomputing a run with its own method must reproduce the on-robot `x, y, course` (check below).

In [ ]:
def relocalize(df, b0, method, r=None, B=None, s_ccw=None, s_cw=None, sign=GYRO_SIGN):
    r = globals()["r"] if r is None else r
    B = globals()["B"] if B is None else B
    s_ccw = S_CCW if s_ccw is None else s_ccw
    s_cw = S_CW if s_cw is None else s_cw
    # Recompute x, y, course with method 'odom' (run A formula) or 'gyro' (run B formula).
    # Starts at the first drive/turn sample, like the robot (rest samples are skipped).
    d = df[df.phase > 0]
    thl = np.radians(d.encL_deg.values); thr = np.radians(d.encR_deg.values)
    path = (thr + thl) * r / 2
    c_odom = (thr - thl) * r / B
    w = sign * (d.gz_mdps.values - b0)
    s = np.where(w >= 0, s_ccw, s_cw)
    inc = w / (1 + s) / 1000 * (d.dt_ms.values / 1000) * np.pi / 180
    c_gyro = np.cumsum(inc)
    course = c_odom if method == "odom" else c_gyro
    dpath = np.diff(np.concatenate([[0.0], path]))
    x = np.cumsum(dpath * np.cos(course)); y = np.cumsum(dpath * np.sin(course))
    return pd.DataFrame({"t": d.t.values, "phase": d.phase.values, "i": d.i.values.astype(int),
                         "x": x, "y": y, "course": course, "path": path,
                         "course_odom": c_odom, "course_gyro": c_gyro})

rec = {}
for name, R in runs.items():
    df = R["df"]
    R["b0_use"] = R["b0"]
    print(f"run {name}: b0 from the 10 s rest = {R['b0']:.1f} mdeg/s = {R['b0']/1000:.3f} deg/s")
    for m in ("odom", "gyro"):
        rec[(name, m)] = relocalize(df, R["b0_use"], m)

# check: own method (with the parameters the robot used, from the header) reproduces the on-robot estimate
for name, own in (("A", "odom"), ("B", "gyro")):
    if own == "gyro" and runs[name]["dt_fixed"]:
        print(f"run {name}: skipped (on-robot gyro course used the wrong dt, see above)"); continue
    H = runs[name]["header"]
    d = runs[name]["df"][runs[name]["df"].phase > 0]
    q = relocalize(runs[name]["df"], runs[name]["b0_use"], own, r=float(H.get("r", r)), B=float(H.get("B", B)),
                   s_ccw=float(H.get("S_CCW", 0)), s_cw=float(H.get("S_CW", 0)))
    ex = np.max(np.hypot(d.x_cm.values - q.x.values, d.y_cm.values - q.y.values))
    ec = np.degrees(np.max(np.abs(wrap(d.course.values - q.course.values))))
    print(f"run {name} recomputed with '{own}': max |dxy| = {ex:.3f} cm, max |dcourse| = {ec:.3f} deg  "
          f"{'OK' if ex < 1 and ec < 0.5 else '<-- check r, B, b0, GYRO_SIGN, S_*'}")

## 4. Trajectories (Step 4.1) and the 4 recomputed paths (Step 4.2)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
ax = axs[0]
ax.plot(X_ref, Y_ref, "k-o", lw=2, ms=4, label="ideal")
for name, col in (("A", "C0"), ("B", "C1")):
    d = runs[name]["df"]; d = d[d.phase > 0]
    ax.plot(d.x_cm, d.y_cm, col, label=f"run {name} (on-robot estimate)")
ax.set_title("Logged x, y (what each robot believed)"); ax.set_aspect("equal"); ax.legend(fontsize=8)
ax.set_xlabel("x, cm"); ax.set_ylabel("y, cm")

ax = axs[1]
ax.plot(X_ref, Y_ref, "k-o", lw=2, ms=4, label="ideal")
sty = {("A","odom"): ("C0","-"), ("A","gyro"): ("C0","--"), ("B","gyro"): ("C1","-"), ("B","odom"): ("C1","--")}
for key, q in rec.items():
    c, ls = sty[key]
    ax.plot(q.x, q.y, color=c, ls=ls, label=f"run {key[0]} recomputed with {key[1]} course")
ax.set_title("4 offline paths, true r and B (dashed = the other sensor)"); ax.set_aspect("equal"); ax.legend(fontsize=8)
ax.set_xlabel("x, cm")
plt.tight_layout(); plt.show()

**Reading these plots.** The controller drives the *estimate* to every waypoint, so the logged `x, y` of both runs lie on the ideal digits almost by construction. The real path of the robot is **not** in its own estimate. The dashed curves (each run recomputed with the *other* sensor) and the floor measurements in Section 6 show where the robot actually went: where the solid and dashed curves of one run split, the two course sources disagree.

## 5. Course and path per segment (Steps 4.3–4.5)

In [ ]:
def segment_stats(name):
    R = runs[name]; own = "odom" if name == "A" else "gyro"; other = "gyro" if own == "odom" else "odom"
    q = rec[(name, own)].copy(); qo = rec[(name, other)]
    dd = R["df"][R["df"].phase > 0]
    q["course"] = dd.course.values                           # the course the robot actually used (logged)
    r_rob = float(R["header"].get("r", r))                   # path as the robot computed it (its r)
    q["path"] = (np.radians(dd.encL_deg.values) + np.radians(dd.encR_deg.values)) * r_rob / 2
    rows = []; p_end_prev = 0.0
    for k in range(1, NSEG + 1):
        seg = q[q.i == k]; drv = seg[seg.phase == 2]
        if drv.empty:
            rows.append(dict(seg=k)); continue
        # mean course on the straight part; unwrap around the continuous reference
        cr = course_ref_cont[k-1]
        c_log = cr + np.mean(wrap(drv.course.values - cr))
        c_oth = cr + np.mean(wrap(qo.loc[drv.index, f"course_{other}"].values - cr))
        p_end = seg.path.values[-1]
        rows.append(dict(seg=k, course=np.degrees(course_ref[k-1] + wrap(c_log - course_ref[k-1])),
                         course_other=np.degrees(course_ref[k-1] + wrap(c_oth - course_ref[k-1])),
                         d_course=np.degrees(wrap(c_log - course_ref[k-1])),
                         d_course_other=np.degrees(wrap(c_oth - course_ref[k-1])),
                         path=p_end - p_end_prev, d_path=p_end - p_end_prev - path_ref[k-1]))
        p_end_prev = p_end
    return pd.DataFrame(rows).set_index("seg")

def dist_to_polyline(px, py):
    P = np.stack([px, py], 1); best = np.full(len(P), np.inf)
    for k in range(NSEG):
        a = np.array([X_ref[k], Y_ref[k]]); b = np.array([X_ref[k+1], Y_ref[k+1]]); ab = b - a
        tt = np.clip(((P - a) @ ab) / (ab @ ab), 0, 1)
        best = np.minimum(best, np.linalg.norm(P - (a + tt[:, None] * ab), axis=1))
    return best

S = {n: segment_stats(n) for n in ("A", "B")}
for n in ("A", "B"):
    for m in ("odom", "gyro"):
        q = rec[(n, m)]; q["dev"] = dist_to_polyline(q.x.values, q.y.values)

In [ ]:
# ---- the comparison table from the Deliverables ----
f1 = lambda v: f"{v:.1f}"
tab = pd.DataFrame({
    "Seg.": ref.seg.astype(str) + " (" + ref.part + ")",
    "course_ref (°)": ref.course_ref_deg.map(f1),
    "course A (°)": S["A"].course.map(f1).values,
    "course B (°)": S["B"].course.map(f1).values,
    "d_course A / B (°)": [f"{a:+.1f} / {b:+.1f}" for a, b in zip(S["A"].d_course, S["B"].d_course)],
    "path_ref (cm)": ref.path_ref_cm.map(f1),
    "path A (cm)": S["A"].path.map(f1).values,
    "path B (cm)": S["B"].path.map(f1).values,
    "d_path A / B (cm)": [f"{a:+.1f} / {b:+.1f}" for a, b in zip(S["A"].d_path, S["B"].d_path)],
})
tot_ref = path_ref.sum(); tA = S["A"].path.sum(); tB = S["B"].path.sum()
tab.loc[len(tab)] = ["Summary", "", "", "",
                     f"mean abs(d_course): {S['A'].d_course.abs().mean():.1f} / {S['B'].d_course.abs().mean():.1f}",
                     f"total {tot_ref:.1f}", f"{tA:.1f}", f"{tB:.1f}",
                     f"{tA-tot_ref:+.1f} / {tB-tot_ref:+.1f}  ({100*(tA-tot_ref)/tot_ref:+.1f} % / {100*(tB-tot_ref)/tot_ref:+.1f} %)"]
display(tab) if "display" in globals() else print(tab.to_string(index=False))
tab.to_csv("lab10_comparison_table.csv", index=False)
try:
    md_table = tab.to_markdown(index=False)
except ImportError:
    md_table = tab.to_string(index=False)
open("lab10_comparison_table.md", "w").write(md_table)

# independent check: course of each run measured by the OTHER sensor
chk = pd.DataFrame({"course_ref (°)": ref.course_ref_deg.values,
                    "run A: gyro course (°)": S["A"].course_other.round(1).values,
                    "run A: d (°)": S["A"].d_course_other.round(1).values,
                    "run B: odom course (°)": S["B"].course_other.round(1).values,
                    "run B: d (°)": S["B"].d_course_other.round(1).values}, index=ref.seg)
print("\nIndependent check: each run's course seen by the other sensor")
display(chk) if "display" in globals() else print(chk.to_string())

In [ ]:
# course(t) and course_odom(t) over the course_ref staircase
fig, axs = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
for ax, name in zip(axs, ("A", "B")):
    q = rec[(name, "odom" if name == "A" else "gyro")]
    stair = np.where(q.i.values >= 1, course_ref_cont[np.clip(q.i.values, 1, NSEG) - 1], 0)
    ax.plot(q.t, np.degrees(stair), "k", lw=2, drawstyle="steps-post", label="course_ref (staircase)")
    ax.plot(q.t, np.degrees(q.course), "C0" if name == "A" else "C1", label="course (used by the robot)")
    ax.plot(q.t, np.degrees(q.course_odom), "C2--", lw=1, label="course_odom")
    ax.plot(q.t, np.degrees(q.course_gyro), "C3:", lw=1, label="course from gyro")
    ax.set_title(f"Run {name}"); ax.set_ylabel("deg"); ax.legend(fontsize=8, loc="best")
axs[-1].set_xlabel("t, s"); plt.tight_layout(); plt.show()

In [ ]:
# error growth vs segment number
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
segs = np.arange(1, NSEG + 1)
ax = axs[0]
ax.plot(segs, S["A"].d_course, "C0o-", label="A: logged course (odom)")
ax.plot(segs, S["B"].d_course, "C1o-", label="B: logged course (gyro)")
ax.plot(segs, S["A"].d_course_other, "C0s--", label="A seen by gyro")
ax.plot(segs, S["B"].d_course_other, "C1s--", label="B seen by odom")
ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("segment"); ax.set_ylabel("d_course, deg"); ax.legend(fontsize=8)
ax.set_xticks(segs)
ax = axs[1]
for (n, m), q in rec.items():
    c, ls = sty[(n, m)]
    dev = q[q.i >= 1].groupby("i").dev.max()
    ax.plot(dev.index, dev.values, color=c, ls=ls, marker="o", label=f"run {n}, {m} course")
ax.set_xlabel("segment"); ax.set_ylabel("max distance to ideal trajectory, cm"); ax.legend(fontsize=8)
ax.set_xticks(segs)
plt.tight_layout(); plt.show()

## 5b. Where the robot really went

Both runs recomputed with the gyro course and the true r, B (the best estimate of the real trajectory), together with the encoder course. If the robot program used the same r, B as this notebook, the scale factor below is 1 and the dotted "ideal × k" line lies on the ideal.

In [ ]:
r_rob = float(runs["A"]["header"].get("r", 2.8)); B_rob = float(runs["A"]["header"].get("B", 15.3))
k_path = r / r_rob
print(f"distance scale real/believed = {r:g}/{r_rob:g} = {k_path:.3f};  "
      f"encoder-course scale used/true = ({r_rob:g}/{B_rob:g}) / ({r:g}/{B:g}) = {(r_rob/B_rob)/(r/B):.3f}")
fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, name in zip(axs, ("A", "B")):
    ax.plot(X_ref, Y_ref, "k-o", lw=2, ms=4, label="ideal (planned)")
    if abs(k_path - 1) > 0.01:
        ax.plot(X_ref * k_path, Y_ref * k_path, "k:", lw=1.5, label=f"ideal x {k_path:.2f}")
    q = rec[(name, "gyro")]; qo = rec[(name, "odom")]
    ax.plot(q.x, q.y, "C1" if name == "B" else "C0", label="real: gyro course, true r")
    ax.plot(qo.x, qo.y, "C2--", lw=1, label="encoder course, true r and B")
    ax.set_title(f"Run {name} ({'odometry' if name == 'A' else 'gyro'} steering)"); ax.set_aspect("equal")
    ax.set_xlabel("x, cm"); ax.legend(fontsize=8)
axs[0].set_ylabel("y, cm"); plt.tight_layout(); plt.show()

rows = []
for name in ("A", "B"):
    q = rec[(name, "gyro")]; qo = rec[(name, "odom")]
    xe, ye = X_ref[-1] * k_path, Y_ref[-1] * k_path
    rows.append({"run": name,
                 "real path (cm)": q.path.values[-1],
                 "planned path (cm)": path_ref.sum(),
                 f"planned x {k_path:.2f} (cm)": path_ref.sum() * k_path,
                 "end vs last waypoint (x k) (cm)": math.hypot(q.x.values[-1] - xe, q.y.values[-1] - ye),
                 "final course, gyro (deg)": np.degrees(q.course.values[-1]),
                 "final course, true encoders (deg)": np.degrees(qo.course_odom.values[-1]),
                 "course_ref of last segment (deg)": np.degrees(course_ref[-1])})
real = pd.DataFrame(rows).set_index("run").round(1)
display(real) if "display" in globals() else print(real.to_string())

## 6. Final pose errors (relative to the last waypoint)

In [ ]:
# Measure on the floor after each run (tape measure from the start mark, protractor for the heading).
# Leave as None if not measured. Coordinates in the same frame as X_ref, Y_ref (start = (0,0), +x = initial heading).
MEASURED = {"A": dict(x=None, y=None, course_deg=None),
            "B": dict(x=None, y=None, course_deg=None)}

rows = []
for name in ("A", "B"):
    d = runs[name]["df"]; last = d.iloc[-1]
    c_last_ref = course_ref[-1]
    row = {"run": name,
           "est. pos. error (cm)": math.hypot(last.x_cm - X_ref[-1], last.y_cm - Y_ref[-1]),
           "est. course error (°)": math.degrees(wrap(last.course - c_last_ref))}
    other = rec[(name, "gyro" if name == "A" else "odom")].iloc[-1]
    row["other-sensor pos. error (cm)"] = math.hypot(other.x - X_ref[-1], other.y - Y_ref[-1])
    row["other-sensor course error (°)"] = math.degrees(wrap(other.course - c_last_ref))
    m = MEASURED[name]
    if m["x"] is not None:
        row["measured pos. error (cm)"] = math.hypot(m["x"] - X_ref[-1], m["y"] - Y_ref[-1])
    if m["course_deg"] is not None:
        row["measured course error (°)"] = (m["course_deg"] - math.degrees(c_last_ref) + 180) % 360 - 180
    rows.append(row)
final = pd.DataFrame(rows).set_index("run").round(2)
display(final) if "display" in globals() else print(final.to_string())

## 7. Which error source dominated? (diagnostics)

In [ ]:
# (a) per-turn comparison of the two course sources: ratio = d(course_odom) / d(course_gyro)
#     constant ratio != 1 for all turns  -> systematic: wrong B (or s); B_eff = B * ratio
#     ratio varies, worse on fast turns   -> slip (or gyro saturation if |gz| ~ W_MAX)
#     ratio_raw uses the gyro without scale-factor correction, ratio_s with S_CCW / S_CW set at the top.
for name in ("A", "B"):
    q = rec[(name, "odom")]
    q0 = relocalize(runs[name]["df"], runs[name]["b0_use"], "odom", s_ccw=0, s_cw=0)
    rows = []
    for k in range(1, NSEG + 1):
        m = ((q.i == k) & (q.phase == 1)).values
        if m.sum() < 3: continue
        do = q.course_odom.values[m][-1] - q.course_odom.values[m][0]
        dg0 = q0.course_gyro.values[m][-1] - q0.course_gyro.values[m][0]
        dg = q.course_gyro.values[m][-1] - q.course_gyro.values[m][0]
        rows.append(dict(seg=k, turn_odom_deg=np.degrees(do), turn_gyro_raw_deg=np.degrees(dg0),
                         turn_gyro_s_deg=np.degrees(dg),
                         ratio_raw=do / dg0 if abs(dg0) > 0.1 else np.nan,
                         ratio_s=do / dg if abs(dg) > 0.1 else np.nan))
    T = pd.DataFrame(rows).set_index("seg").round(3)
    print(f"Run {name}: turns  odom/gyro = {T.ratio_raw.mean():.3f} (no s), {T.ratio_s.mean():.3f} (with S_CCW/S_CW);  "
          f"B_eff = B * ratio = {B * T.ratio_raw.mean():.2f} cm (no s), {B * T.ratio_s.mean():.2f} cm (with S_CCW/S_CW)")
    display(T) if "display" in globals() else print(T.to_string())

# (b) gyro saturation, (c) bias: difference of the two b0 values = how much b0 changes between runs
for name in ("A", "B"):
    d = runs[name]["df"]; b0 = runs[name]["b0_use"]
    wdeg = np.abs(d.gz_mdps - b0) / 1000
    print(f"run {name}: peak |gz - b0| = {wdeg.max():.0f} deg/s (W_MAX = {W_MAX:.0f}) "
          f"{'<-- SATURATED' if wdeg.max() > 0.95 * W_MAX else 'ok'}")
db0 = (runs["A"]["b0_use"] - runs["B"]["b0_use"]) / 1000
print(f"b0(A) - b0(B) = {db0:+.4f} deg/s; a bias error of this size gives {db0 * runs['B']['df'].t.iloc[-1]:+.2f} deg over run B")

# (d) path: same encoders in both runs -> total path error should be similar in A and B
print(f"total path error: A {100*(tA-tot_ref)/tot_ref:+.2f} %,  B {100*(tB-tot_ref)/tot_ref:+.2f} %")

## 8. Short analysis (≤ 1 page)

*Write your own answers from the numbers above.*

1. **Course vs. path.** Which run is closer to the ideal trajectory in course, and which in path? Why are the path errors similar (both runs take `path` from the same encoders, and turn-then-drive makes each segment end inside the acceptance radius) while the course errors are not (different course source)? Use the *independent check* table and the floor measurements, not only the logged course, which the controller holds at `course_ref` by design. Note that every segment ends anywhere inside the acceptance radius `THRESHOLD`, so `d_path` of about −THRESHOLD per segment and a few degrees of `d_course` on short segments come from the controller, not from the sensors.
2. **Dominant error source in each run.** Slip, `B`, bias or scale factor? Support it with Section 7: per-turn ratio (constant → `B`; varies with turn rate → slip), peak `|gz|` vs `W_MAX` (saturation), residual bias in the final rest, and the 4 recomputed paths from Section 4.
3. **Combining `course_odom` and `gyro_inc`.** Which one to trust at high frequency (turns) and which at low frequency (long straight segments, drift)? This is the complementary filter of Lab 11: `course = a * course_odom + (1 - a) * (course + gyro_inc)`. Use the recomputation function above to try a few values of `a` offline.